# Notebook 04 · Evaluación final: Precision, Recall, F1 y matriz de confusión sobre Test

| Redes Neuronales III | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-III` |


**Problema:** MNIST, 10 clases. El modelo evaluado es el **regularizado** del Notebook 03
(L2 + Dropout + Early Stopping), entrenado sobre la muestra de 12 000 imágenes.

> **Alcance:** usar **Test por primera y única vez** para reportar el desempeño final con métricas que
> van más allá del accuracy: Precision, Recall y F1 macro, reporte por clase, matriz de confusión,
> identificación de la confusión más frecuente y visualización de un error real.

---

## 1. Por qué el accuracy no basta

Con clases balanceadas el accuracy resume bien, pero oculta comportamientos por clase. Las métricas
por clase responden preguntas distintas:

| Métrica | Pregunta que responde | Fórmula |
|---|---|---|
| Accuracy | ¿Qué proporción total acertó? | (TP+TN) / total |
| Precision | Cuando predije la clase, ¿cuántas veces acerté? | TP / (TP+FP) |
| Recall | De los ejemplos reales de la clase, ¿cuántos recuperé? | TP / (TP+FN) |
| F1 | ¿Cómo equilibrio Precision y Recall? | Media armónica de ambas |

En MNIST se reporta el **promedio macro**: se calcula la métrica por dígito y se promedia dando el mismo
peso a cada clase. La **matriz de confusión** completa el cuadro: filas = clase real, columnas = clase
predicha; la diagonal son aciertos y el resto son confusiones concretas.

## 2. Entorno, datos y reconstrucción del modelo final

El notebook reentrena el modelo regularizado con la misma semilla y el mismo procedimiento del
Notebook 03 (muestra de 12 000, L2, Dropout y parada temprana), de modo que sea autocontenido: puede
ejecutarse sin depender de sesiones anteriores de Colab.

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
import pandas as pd
import random, gc

random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

I0000 00:00:1791728399.944964     530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791728402.108397     530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1


In [2]:
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

(x_train_full, y_train_full), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

# División estratificada Train / Dev (80 / 20). Test queda intocable hasta el final.
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full,
)

# Test se normaliza completo (31 MB); Train y Dev se normalizan por lotes para
# mantener el consumo de memoria del kernel bajo control.
x_test = x_test_raw.astype("float32") / 255.0


def normalizar_lote(imagenes):
    """Convierte un lote uint8 (n, 28, 28) en float32 (n, 28, 28) dentro de [0, 1].

    El aplanado a 784 características lo realiza la capa Flatten del modelo.
    """
    return imagenes.astype("float32") / 255.0


print("Train:", x_train.shape, "| Dev:", x_dev.shape, "| Test:", x_test.shape)

x_train_reg, _, y_train_reg, _ = train_test_split(
    x_train, y_train, train_size=12000, random_state=SEED, stratify=y_train
)
print()
print("Muestra de entrenamiento del modelo final:", x_train_reg.shape)

Train: (48000, 28, 28) | Dev: (12000, 28, 28) | Test: (10000, 28, 28)

Muestra de entrenamiento del modelo final: (12000, 28, 28)


## 3. Entrenamiento del modelo regularizado (réplica del Notebook 03)

In [3]:
tf.random.set_seed(42)
modelo_reg = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.30),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.30),
    layers.Dense(10, activation="softmax"),
])
modelo_reg.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

perdida_fn = keras.losses.SparseCategoricalCrossentropy()
optimizador = modelo_reg.optimizer


@tf.function
def paso(xb, yb):
    with tf.GradientTape() as cinta:
        logits = modelo_reg(xb, training=True)
        perdida = perdida_fn(yb, logits)
    g = cinta.gradient(perdida, modelo_reg.trainable_variables)
    optimizador.apply_gradients(zip(g, modelo_reg.trainable_variables))
    return perdida, tf.argmax(logits, axis=1)


def dev_loss_acc():
    pt, ac, tot = 0.0, 0, 0
    for inicio in range(0, len(x_dev), 1000):
        xb = normalizar_lote(x_dev[inicio:inicio + 1000])
        yb = y_dev[inicio:inicio + 1000]
        probs = modelo_reg(xb, training=False).numpy()
        pt += float(perdida_fn(yb, probs)) * len(yb)
        ac += int((probs.argmax(axis=1) == yb).sum())
        tot += len(yb)
    return pt / tot, ac / tot


indices = np.arange(len(x_train_reg))
mejor_dev, mejores_pesos, sin_mejora, epocas_reg = np.inf, None, 0, 0
for epoch in range(20):
    np.random.shuffle(indices)
    for inicio in range(0, len(indices), 64):
        idx = indices[inicio:inicio + 64]
        paso(normalizar_lote(x_train_reg[idx]), y_train_reg[idx])
    dl, _ = dev_loss_acc()
    epocas_reg = epoch + 1
    if dl < mejor_dev - 1e-4:
        mejor_dev, sin_mejora = dl, 0
        mejores_pesos = [v.numpy().copy() for v in modelo_reg.trainable_variables]
    else:
        sin_mejora += 1
        if sin_mejora >= 2:
            break
    gc.collect()

for v, peso in zip(modelo_reg.trainable_variables, mejores_pesos):
    v.assign(peso)
print("Épocas ejecutadas:", epocas_reg, "| pesos restaurados a la mejor época (Dev loss =",
      round(mejor_dev, 4), ")")

Épocas ejecutadas: 8 | pesos restaurados a la mejor época (Dev loss = 0.1326 )


## 4. Predicciones sobre Test (uso único y final)

A partir de este punto **no se ajusta nada más**: Test solo se usa para reportar. Las predicciones se
generan por lotes de 1 000 imágenes para mantener el consumo de memoria bajo control.

In [4]:
probabilidades_test = []
for inicio in range(0, len(x_test), 1000):
    lote = modelo_reg.predict(x_test[inicio:inicio + 1000], verbose=0)
    probabilidades_test.append(lote)
probabilidades_test = np.vstack(probabilidades_test)
y_pred = np.argmax(probabilidades_test, axis=1)

print("Predicciones sobre Test:", y_pred.shape)
print("Primeras 20 predicciones:", y_pred[:20].tolist())
print("Primeras 20 clases reales:", y_test[:20].tolist())
print("Coincidencias en la muestra:", int((y_pred[:20] == y_test[:20]).sum()), "de 20")

Predicciones sobre Test: (10000,)
Primeras 20 predicciones: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4]
Primeras 20 clases reales: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4]
Coincidencias en la muestra: 20 de 20


## 5. Accuracy, Precision, Recall y F1 (promedio macro)

In [5]:
accuracy = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

metricas = pd.DataFrame({
    "metrica": ["Accuracy", "Precision macro", "Recall macro", "F1 macro"],
    "resultado": [round(accuracy, 4), round(precision_macro, 4),
                  round(recall_macro, 4), round(f1_macro, 4)],
})
print("=== MÉTRICAS FINALES SOBRE TEST ===")
display(metricas)
print(metricas.to_string(index=False))
print()
print("Accuracy   :", round(accuracy, 4))
print("Precision  :", round(precision_macro, 4))
print("Recall     :", round(recall_macro, 4))
print("F1 macro   :", round(f1_macro, 4))

=== MÉTRICAS FINALES SOBRE TEST ===


,metrica,resultado
0,Accuracy,0.9644
1,Precision macro,0.9643
2,Recall macro,0.9640
3,F1 macro,0.9641


        metrica  resultado
       Accuracy     0.9644
Precision macro     0.9643
   Recall macro     0.9640
       F1 macro     0.9641

Accuracy   : 0.9644
Precision  : 0.9643
Recall     : 0.964
F1 macro   : 0.9641


## 6. Reporte de clasificación por dígito

El reporte permite identificar clases débiles: se busca al menos un dígito cuyo Precision, Recall o F1
quede por debajo del promedio macro, para interpretarlo en el análisis.

In [6]:
reporte = classification_report(y_test, y_pred, digits=4, zero_division=0)
print(reporte)

lineas = [l.split() for l in reporte.splitlines()
          if l.strip() and l.strip()[0].isdigit()]
peor_f1, peor_digito = 1.0, None
for partes in lineas:
    f1_clase = float(partes[3])
    if f1_clase < peor_f1:
        peor_f1, peor_digito = f1_clase, partes[0]
print()
print(f"Clase con menor F1: dígito {peor_digito} (F1 = {peor_f1:.4f}) "
      f"frente al macro {f1_macro:.4f}")

              precision    recall  f1-score   support

           0     0.9766    0.9796    0.9781       980
           1     0.9732    0.9930    0.9830      1135
           2     0.9676    0.9535    0.9605      1032
           3     0.9551    0.9693    0.9622      1010
           4     0.9710    0.9552    0.9630       982
           5     0.9680    0.9507    0.9593       892
           6     0.9549    0.9718    0.9633       958
           7     0.9745    0.9650    0.9697      1028
           8     0.9361    0.9620    0.9489       974
           9     0.9664    0.9395    0.9528      1009

    accuracy                         0.9644     10000
   macro avg     0.9643    0.9640    0.9641     10000
weighted avg     0.9645    0.9644    0.9644     10000


Clase con menor F1: dígito 8 (F1 = 0.9489) frente al macro 0.9641


## 7. Matriz de confusión

In [7]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6.0, 5.0))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))
disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format="d", xticks_rotation=0)
ax.set_title("Figura 6. Matriz de confusión - MNIST (Test, 10 000 imágenes)")
plt.tight_layout(); plt.savefig("figIII_05_confusion.png"); plt.show()
plt.close(fig); _ = gc.collect()

print("Total de errores en Test:", int((y_pred != y_test).sum()), "de", len(y_test))

Total de errores en Test: 356 de 10000


## 8. La confusión más frecuente y un error real

Se anula la diagonal de la matriz y se localiza el par (real, predicho) con más casos. Después se
visualiza una imagen concreta de ese par: el análisis profesional no esconde los errores, los explica.

In [8]:
cm_errores = cm.copy()
np.fill_diagonal(cm_errores, 0)

real_confundido, predicho_como = np.unravel_index(np.argmax(cm_errores), cm_errores.shape)
cantidad = int(cm_errores[real_confundido, predicho_como])

print(f"La confusión más frecuente fue: {real_confundido} clasificado como "
      f"{predicho_como} ({cantidad} veces)")

pares = sorted([(cm_errores[r, c], r, c) for r in range(10) for c in range(10)], reverse=True)
print()
print("Top 5 confusiones (real -> predicho, casos):")
for cant, r, c in pares[:5]:
    print(f"   {r} -> {c} : {int(cant)}")

La confusión más frecuente fue: 4 clasificado como 9 (20 veces)

Top 5 confusiones (real -> predicho, casos):
   4 -> 9 : 20
   9 -> 8 : 14
   9 -> 4 : 14
   5 -> 8 : 14
   9 -> 3 : 13


In [9]:
indices_error = np.where((y_test == real_confundido) & (y_pred == predicho_como))[0]

fig, axes = plt.subplots(1, min(6, len(indices_error)), figsize=(9.5, 2.4))
axes = np.atleast_1d(axes)
for ax, idx in zip(axes, indices_error[:6]):
    ax.imshow(x_test_raw[idx], cmap="gray")
    ax.set_title(f"Real {y_test[idx]}\nPred {y_pred[idx]}", fontsize=8, color="#c0392b")
    ax.axis("off")
fig.suptitle(f"Figura 7. Ejemplos reales de la confusión {real_confundido} -> {predicho_como}",
             y=1.08)
plt.tight_layout(); plt.savefig("figIII_06_error_muestra.png"); plt.show()
plt.close(fig); _ = gc.collect()

idx0 = int(indices_error[0])
print("Imagen analizada: índice", idx0,
      "| confianza del modelo:", round(float(probabilidades_test[idx0].max()), 4))

Imagen analizada: índice 571 | confianza del modelo: 0.6726


## 9. Análisis y conclusiones

- El **accuracy** resume el desempeño global, pero el **reporte por clase** muestra dónde se concentra
  el error residual: los dígitos con trazos ambiguos o similares entre sí suelen tener menor Recall.
- **Precision y Recall macro** cercanos al accuracy indican que el error está repartido y no
  concentrado en una sola clase; una caída fuerte de Recall en un dígito señalaría que el modelo lo
  confunde sistemáticamente con otro.
- La **matriz de confusión** convierte el error agregado en pares concretos (por ejemplo 5→3 o 8→3),
  que es la información accionable para decidir mejoras: más datos de esas clases, aumento de datos o
  arquitecturas que capturen forma local.
- Visualizar un error real cierra el ciclo: la imagen suele ser efectivamente ambigua incluso para un
  observador humano, lo que contextualiza el límite del modelo.
- **Test se usó una sola vez.** Cualquier ajuste posterior basado en estos números invalidaría la
  estimación: para eso existe Dev.

## 10. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
- LeCun, Y., Cortes, C., & Burges, C. J. C. (2010). *MNIST handwritten digit database*. http://yann.lecun.com/exdb/mnist/
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/